In [ ]:
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001.wav
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/segtools.py
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001.seg_B1
!wget -q https://phonetics-spbu.github.io/courses/python_textbook/files/cta0001.seg_G1


Часто информация о частоте основного тона (ЧОТ) в сигнале хранится в виде меток, которые обозначают границы периодов основного тона (ОТ). Заметьте, что принципы расставления границ могут отличаться: у нас началом периода принято считать переход через 0 в положительную область, в других местах &mdash; минимальное значение (отрицательный пик) или максимальное (положительный).

### Основные форматы записи меток основного тона

#### Формат SEG_G1 (Wave Assistant)

Wave Assistant расставляет границы периодов основного тона на уровне G1. На этом уровне часто (но не всегда!) также находятся метки, обозначающие границы файла. Они не являются метками границ периодов ОТ.

#### Формат PointProcess (Praat)

Praat также позволяет получить информацию о границах периодов основного тона. Для этого можно воспользоваться командой &ldquo;Sound: To PointProcess (periodic, cc)...&rdquo;. Границы периодов ставятся по положительным пикам. Получившийся объект называется PointProcess и может быть сохранён в текстовом файле. Его типичная структура такова:

Время каждой метки хранится в виде вещественного числа и выражено в секундах. Кроме меток, файл содержит также информацию о времени начала и конца файла (также в секундах) и количестве меток.

#### Формат PM (REAPER)

Утилита [REAPER](https://github.com/google/REAPER), позволяющая с довольно высокой точностью получать границы периодов основного тона, также даёт информацию о границах периодов. Здесь они ставятся по отрицательным пикам.

Если вы работаете в Unix-подобной системе, REAPER можно легко установить с GitHub. Для этого запустим следующий код. Если вы делаете это в Google Colab, первой строчкой ячейки должна быть %%bash, чтобы показать, что это консольные команды. Если вы работаете на локальной машине, просто вводите все строчки, начиная со второй, в терминал (перед этим перейдите в удобную директорию).

Чтобы запустить REAPER в Google Colab, запустите такую ячейку:

In [ ]:
%%bash
REAPER/build/reaper -i cta0001.wav -p cta0001.pm -a

Для запуска на локальной машине введите в консоль такую команду:

```bash
REAPER/build/reaper -i cta0001.wav -p cta0001.pm -a
```

Параметр `-i` указывает на имя входного файла, `-p` &mdash; на имя выходного файла с метками, `-a` &mdash; указывает программе писать результат в виде текстового файла.

Структура файла такова:

```
EST_File Track
DataType ascii
NumFrames 271
NumChannels 1
FrameShift 0.00000
VoicingEnabled true
EST_Header_End
0.004580 1 0.000000
0.011293 1 0.000000
0.017687 1 0.000000
0.024263 1 0.000000
0.030884 1 0.000000
0.037415 1 0.000000
```

В поле `NumFrames` содержится количество меток, после конца заголовка `EST_Header_End` начинаются собственно данные, где первое поле &mdash; время метки в секундах, второе поле содержит 1, если фрейм звонкий, и 0, если он глухой. Третье поле содержит частоту основного тона, которая в этом режиме не вычисляется.

С параметром `-f` REAPER сгенерирует аналогичный файл, но содержащий не границы периодов, а значения частоты основного тона с равным шагом (для таких файлов предлагается использовать расширение .f0).

```
EST_File Track
DataType ascii
NumFrames 345
NumChannels 1
FrameShift 0.00000
VoicingEnabled true
EST_Header_End
0.000000 0 -1.000000
0.005000 1 143.181824
0.010000 1 156.382980
0.015000 1 156.382980
0.020000 1 169.615387
0.025000 1 155.281693
```

На глухих участках значение ЧОТ всегда будет равно &minus;1.0.

### Обработка меток периодов

Вспомним, что часто для извлечения информации из меток необходимо обрабатывать их парами. Метки границ периодов ОТ &mdash; не исключение. Если мы знаем положение двух соседних меток, мы легко можем вычислить длину соответствующего периода как разность из позиций, а из длины периода &mdash; значение ЧОТ как обратную ей величину.

Не любая пара меток в файле соответствует периоду ОТ. Некоторые из них отмечают глухие участки:

![](https://phonetics-spbu.github.io/courses/python_textbook/images/voiceless_part.png)

Отличить одно от другого мы можем, задав минимальное возможное значение ЧОТ. Тогда мы сможем отличить такие интервалы по длине: периодами ОТ будут только те интервалы, которые короче определённого порога. Таким парам меток удобно сопоставить значение NaN. NaN (Not a Number) &mdash; значение, которое часто используют в качестве заместителя для пропущенных данных. Его можно получить, например, так:

In [ ]:
nan_val = float("nan")
print(nan_val)

Или так:

In [ ]:
import numpy as np
nan_val2 = np.nan
print(nan_val2)

NaN не равен никакому числу и не равен сам себе:

In [ ]:
nan_val == nan_val

Чтобы проверить, является ли значение NaN, есть специальные функции:

In [ ]:
np.isnan(nan_val)

Если мы будем строить графики по данным, содержащим NaN, с помощью matplotlib, то в соответствующих местах появятся разрывы. Сравните:

In [ ]:
x = list(range(10))
y1 = [1, 2, 3, np.nan, np.nan, 5, 4, 3, 2, 1]
y2 = [1, 2, 3, 4, 5, 5, 4, 3, 2, 1]

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2)
axes[0].plot(x, y1)
axes[1].plot(x, y2)

![](https://phonetics-spbu.github.io/courses/python_textbook/images/nan_plot.png)

### Практические задания

#### 4.1. Построение графика ЧОТ из файла SEG_G1

Напишите функцию, которая принимает на вход имя файла SEG и минимальное значение ЧОТ, а возвращает пару из двух списков: список позиций каждого периода (за позицию периода возьмите его середину) и список значений ЧОТ в каждом периоде. Глухим участкам сопоставьте значение NaN.

Алгоритм выполнения:

In [ ]:
def get_f0(filename: str, min_f0: float = 50.0) -> tuple[list[float], list[float]]:
    times, f0_values = [], []
    # прочитать сег
    # убрать из него метки начала и конца файла
    # перебрать метки попарно
    # в каждом интервале:
    # определить место середины В СЕКУНДАХ, добавить в times
    # определить значение ЧОТ
    # если оно >= минимального, добавить в f0_values
    # в противном случае добавить в f0_values NaN
    return times, f0_values

Используйте эту функцию для построения графика несглаженной ЧОТ:

In [ ]:
times, f0_values = get_f0("cta0001.seg_G1")
plt.plot(times, f0_values)
plt.show()

#### 4.2. Сглаживание мелодического контура

Напишите функцию, которая принимает на вход список позиций точек и значений ЧОТ в них (возвращаемых функцией из предыдущего задания) и сглаживает значения ЧОТ методом скользящего среднего: значение каждой точки должно стать средним арифметическим между её соседями в пределах окна, заданного в миллисекундах (передаётся в качестве аргумента функции). Значения NaN и значения из других звонких участков в усреднении участвовать не должны.

#### 4.3. Определение границ звонких участков

Напишите программу, которая определяет границы каждого звонкого участка в файле по меткам ОТ и записывает их в файл SEG_G2. Названием каждого метки сделайте число, которое показывает разницу между максимальным и минимальным значением ЧОТ в этом участке. Выразите эту разницу в полутонах. Формула для перевода разницы между двумя частотами в полутоны:

$$ st = 12 \cdot \log_{2}{\frac{f_2}{f_1}} $$

Функции для логарифма:

In [ ]:
from math import log2
log2(10)

In [ ]:
import numpy as np
np.log2(10)

Возможный алгоритм:

In [ ]:
def voiced_regions(seg_filename: str, min_f0: float = 50) -> None:
    # прочитаем метки из файла

    # создадим список, куда будем добавлять наши звонкие участки
    # каждый звонкий участок характеризуется тремя параметрами:
    # время начала, время конца, список значений ЧОТ внутри него
    # можно реализовать это в виде словаря с тремя ключами,
    # два из которых будут хранить числовые значения, третий - список чисел

    # сразу добавим в этот список первый участок
    # его время начала соотвествует первой метке
    # а время конца мы ещё не знаем

    # переберём все пары меток
    # если пара является настоящим периодом ОТ, то добавляем значение ЧОТ в текущий участок
    # (т.е. последний участок в большом списке)
    # иначе текущему участку назначаем время конца, равное левой метке в паре
    # и добавляем в большой список новый участок со временем начала, равным правой метке в паре

    # после конца цикла последнему участку назначим время конца, равное последней метке

    # заведём список новых меток
    # переберём все участки
    # для каждого вычислим перепад ЧОТ, переведём в строку
    # создадим метку начала на уровне G2 с таким названием и метку конца с пустым названием 

    # запишем все созданные метки в файл с необходимыми параметрами
    pass

#### 4.4. Построение графика мелодической деклинации

Напишите функцию, которая принимает на вход имя звукового файла и сопутствующих меток и изображает график мелодической деклинации, т.е. максимумы ЧОТ в ударном гласном каждого слова, для каждой синтагмы. По оси X должны быть отмечены ударные гласные, по оси Y &mdash; значение максимума ЧОТ в ударном гласном. Сделайте функции опциональный аргумент, в котором будет передаваться относительная частота для перевода в шкалу полутонов. Если он будет равен `None`, оставьте частоту в Гц.

#### 4.5. Построение графиков ЧОТ разных интонационных моделей

Напишите программу, которая обрабатывает все файлы в архиве *cta0001-0010.zip* и рисует N графиков (по количеству разных интонационных моделей, встретившихся в материале), на каждом из которых изображены все интонационные кривые внутри ядерного гласного из всех синтагм, оформленных этой моделью. Ядром по умолчанию считается ударный слог последнего слова в синтагме. Если это не так, перед словом, содержащим ядерный слог, ставится знак [-].

#### 4.6. Сортировка звуков по ЧОТ

Напишите программу, которая читает файл WAV и параллельные ему файлы SEG_G1 и SEG_B1 и склеивает звуки файла в следующем порядке: сначала все глухие, потом все звонкие в порядке возрастания максимальной ЧОТ в них. Паузы в итоговый сигнал не включайте.